# Tracklet classifier

**Goal:** score every linked tracklet: real moving object or chance alignment / near miss? We learn the tree family
(decision tree → random forest → gradient boosting), cross-validate **by night**, and pick an operating threshold.

Read `docs/tracklet_classifier.md` alongside this notebook.

In [ ]:
import json
import subprocess
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
from sklearn.model_selection import GroupKFold
from sklearn.tree import DecisionTreeClassifier, plot_tree

from asthunt.config.load_config import load_config
from asthunt.dataset.make_unit_tag import make_unit_tag
from asthunt.acquire.load_unit_metadata import load_unit_metadata
from asthunt.features.compute_tracklet_features import FEATURE_COLUMNS
from asthunt.models.train_baseline_classifier import train_baseline_classifier
from asthunt.models.train_tracklet_classifier import train_tracklet_classifier
from asthunt.evaluate.compute_classification_metrics import compute_classification_metrics
from asthunt.evaluate.choose_threshold import choose_threshold
from asthunt.evaluate.check_split_leakage import check_split_leakage

project = load_config("project"); paths = project["paths"]
acquire = load_config("acquire"); nights, field = acquire["nights"], acquire["field"]


def unit_of(night):
    """The original quadrant (configs/acquire.yaml: field, ccd, quadrant) on one night."""
    return {"field": acquire["field"], "ccd": acquire["ccd"], "quadrant": acquire["quadrant"], "night": night}


def load_tracklets(split_nights):
    return pd.concat([pd.read_parquet(paths["processed"] / f"tracklets__{make_unit_tag(unit_of(n))}.parquet") for n in split_nights], ignore_index=True)

train, validation = load_tracklets(nights["train"]), load_tracklets(nights["validation"])
print("leakage:", check_split_leakage(train["night"], validation["night"]))
for name, table in [("train", train), ("validation", validation)]:
    print(f"{name:10s}: {len(table):5d} tracklets, {table['label'].sum():4d} real, {(table['label'] == 0).sum():4d} bogus")

## 1. Explore the features

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(22, 7)); axes = axes.ravel()
for ax, feature in zip(axes, FEATURE_COLUMNS):
    values = train[feature].replace([np.inf, -np.inf], np.nan).dropna()
    bins = np.linspace(*np.percentile(values, [1, 99]), 40)
    for label, name in [(0, "bogus"), (1, "real")]:
        ax.hist(train.loc[train["label"] == label, feature].clip(bins[0], bins[-1]), bins=bins, alpha=0.6, density=True, label=name)
    ax.set_title(feature)
axes[0].legend(); axes[-1].axis("off"); plt.show()

## 2. One decision tree: readable rules
A depth-3 tree asks at most 3 yes/no questions. Read the rules it found.

In [ ]:
X_train, y_train = train[FEATURE_COLUMNS].to_numpy(), train["label"].to_numpy()
X_val, y_val = validation[FEATURE_COLUMNS].to_numpy(), validation["label"].to_numpy()

tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
plt.figure(figsize=(20, 7))
plot_tree(tree, feature_names=FEATURE_COLUMNS, class_names=["bogus", "real"], filled=True, impurity=False, proportion=True, fontsize=9)
plt.show()

## 3. Compare model families with cross-validation **by night**
`GroupKFold` keeps all tracklets of a night in the same fold, so the model is always scored on unseen nights.

In [ ]:
models = {
    "logistic regression": lambda: train_baseline_classifier,
    "decision tree (depth 3)": lambda: (lambda X, y: DecisionTreeClassifier(max_depth=3, random_state=0).fit(X, y)),
    "random forest": lambda: (lambda X, y: RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0).fit(X, y)),
    "gradient boosting (XGBoost)": lambda: train_tracklet_classifier,
}
X_filled = np.nan_to_num(X_train)                       # logistic regression can't take NaN (e.g. std of 1 value)
folds = GroupKFold(n_splits=4)
results = {}
for name, make in models.items():
    scores = []
    for fit_rows, test_rows in folds.split(X_filled, y_train, groups=train["night"]):
        model = make()(X_filled[fit_rows], y_train[fit_rows])
        scores.append(average_precision_score(y_train[test_rows], model.predict_proba(X_filled[test_rows])[:, 1]))
    results[name] = scores
    print(f"{name:28s} AP per fold {np.round(scores, 3)}  mean {np.mean(scores):.3f}")

Look at the **spread across folds**, not only the mean. If two models' fold scores overlap, the difference may be noise.

## 4. Final model: gradient boosting on all training nights, judged on validation

In [ ]:
model = train_tracklet_classifier(np.nan_to_num(X_train), y_train, seed=project["seed"])
validation_scores = model.predict_proba(np.nan_to_num(X_val))[:, 1]
print(f"validation AP: {average_precision_score(y_val, validation_scores):.3f}")

importance = pd.Series(model.feature_importances_, index=FEATURE_COLUMNS).sort_values()
importance.plot.barh(figsize=(6, 4), title="feature importance (gain share)"); plt.show()

## 5. Operating threshold
A real tracklet thrown away is a lost discovery; a bogus tracklet costs a human one look. We keep **95% of real
tracklets** on validation, then count how many bogus tracklets per night a person would have to vet.

In [ ]:
tracklet_threshold = choose_threshold(y_val, validation_scores, target_recall=0.95)
m = compute_classification_metrics(y_val, validation_scores, tracklet_threshold)
print(f"threshold {tracklet_threshold:.3f}: recall {m['recall']:.3f}, precision {m['precision']:.3f}, "
      f"bogus tracklets per night {m['false_positives'] / len(nights['validation']):.1f}")

## 6. Where does it fail? Recall by magnitude

In [ ]:
validation["score"] = validation_scores
real = validation[validation["label"] == 1].copy()
real["kept"] = real["score"] >= tracklet_threshold
by_mag = real.groupby(pd.cut(real["mag_mean"], np.arange(16.5, 22.5, 0.5)), observed=True)["kept"].agg(["mean", "size"])
print(by_mag.round(2).to_string())

## 7. Save the model and threshold

In [ ]:
git_sha = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=paths["models"].parent).stdout.strip()
model_name = f"tracklet_classifier__{date.today():%Y%m%d}__{git_sha}"
model.save_model(paths["models"] / f"{model_name}.json")

thresholds_file = paths["processed"] / "thresholds.json"
thresholds = json.loads(thresholds_file.read_text())
thresholds["tracklet"] = tracklet_threshold
thresholds_file.write_text(json.dumps(thresholds, indent=1))
print("saved", model_name, "and thresholds", thresholds)

## Exercises
1. Drop `score_mean` and `score_min` (the CNN features). How much does validation AP fall? What does that say about the CNN?
2. Train XGBoost with `max_depth=8`. Does cross-validated AP improve, or does the fold spread grow?
3. Replace `GroupKFold` with plain `KFold(shuffle=True)`. Are the scores higher? Why is that number untrustworthy?